# Adaptive RAG Experimental Notebook

This notebook documents the full experimental pipeline used in our project. The structure mirrors the final report, progressing from setup and method definition to evaluation and result analysis.


## Introduction and Experiment Overview

# Run everything: Dataset -> Corpus -> Indexing -> RAG → Adaptive RAG -> (optional) DPO → Eval

This notebook is designed for RunPod where your code lives in `/workspace`. 

Set `USE_OPENAI=True` if you want OpenAI for generation/rewriting/judging. Otherwise the notebook uses a local HF instruct model for generation and (optionally) DPO.


In [1]:
# Initializing retrieval indexes and search components
!pip install rank-bm25 joblib


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


## Environment Setup and Imports

This notebook includes:
- Retrieval quality: Recall@K, MRR@K
- Answer quality: normalized EM, F1, ROUGE-L (optional), SupportOverlap (answer-in-evidence)
- Efficiency: latency p50/p95, throughput

**Important:** this requires the updated `workspace/data/build_corpus.py`, `workspace/data/build_benchmark.py`, and `workspace/evaluation/eval_runner.py` implementations that compute `gold_doc_id` (sha1 of normalized context) and record before/after stats.


In [2]:
# Importing libraries and shared utilities
import sys
from pathlib import Path

# Find repo root by walking up until we see the "workspace" folder
p = Path.cwd().resolve()
while p != p.parent and not (p / "workspace").exists():
    p = p.parent

assert (p / "workspace").exists(), f"Couldn't find repo root above {Path.cwd().resolve()}"

REPO_ROOT = p
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Ensure package init files exist
(REPO_ROOT / "workspace" / "__init__.py").touch(exist_ok=True)
for sub in ["data", "retrieval", "models", "rlhf", "evaluation", "utils"]:
    (REPO_ROOT / "workspace" / sub / "__init__.py").touch(exist_ok=True)

print("✅ REPO_ROOT =", REPO_ROOT)
print("✅ sys.path[0] =", sys.path[0])


✅ REPO_ROOT = /workspace
✅ sys.path[0] = /workspace


In [ ]:
# Importing libraries and shared utilities (YOU MUST ADD AN API KEY)
import os
from pathlib import Path

os.environ["OPENAI_API_KEY"] = "YOUR-API-KEY HERE"

ARTIFACTS = REPO_ROOT / "artifacts"
DATA_DIR = ARTIFACTS / "benchmarks"
CORPUS_DIR = ARTIFACTS / "corpus"
INDEX_DIR = ARTIFACTS / "indexes"
EVAL_DIR = ARTIFACTS / "eval_runs"
PREFS_DIR = ARTIFACTS / "prefs"
MODELS_DIR = ARTIFACTS / "models"

for p in [DATA_DIR, CORPUS_DIR, INDEX_DIR, EVAL_DIR, PREFS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Artifacts dir:", ARTIFACTS)

HF_ID = "virattt/financial-qa-10K"

# Embeddings model
EMBED_MODEL = "BAAI/bge-small-en-v1.5"

# Local generator model (use for baseline + DPO)
LOCAL_GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"

# Optional OpenAI usage
USE_OPENAI = bool(os.getenv("OPENAI_API_KEY"))  # auto-enable if key exists
OPENAI_MODEL = "gpt-4o-mini"

print("USE_OPENAI =", USE_OPENAI)


Artifacts dir: /workspace/artifacts
USE_OPENAI = True


In [ ]:
import sys
from pathlib import Path

# Add repo root to PYTHONPATH
REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Added to PYTHONPATH:", REPO_ROOT)


Added to PYTHONPATH: /workspace/notebooks


In [ ]:
# Downloading All Dependencies
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# Downloads dataset splits to parquet
from workspace.data.download_dataset import download_financial_qa_10k

download_financial_qa_10k(
    hf_id=HF_ID,
    out_dir=str(DATA_DIR),
    splits=("train","test"),
)

train_parquet = DATA_DIR / f"{HF_ID.replace('/','__')}_train.parquet"
test_parquet  = DATA_DIR / f"{HF_ID.replace('/','__')}_test.parquet"
train_parquet, test_parquet


2025-12-14 22:55:57,900 | data.download_dataset | INFO | Available splits for virattt/financial-qa-10K: ['train']
2025-12-14 22:55:57,902 | data.download_dataset | INFO | Downloading virattt/financial-qa-10K split=train
2025-12-14 22:55:58,495 | data.download_dataset | INFO | Saved train -> /workspace/artifacts/benchmarks/virattt__financial-qa-10K_train.parquet
2025-12-14 22:55:58,496 | data.download_dataset | WARNING | Split 'test' not found. Skipping.


(PosixPath('/workspace/artifacts/benchmarks/virattt__financial-qa-10K_train.parquet'),
 PosixPath('/workspace/artifacts/benchmarks/virattt__financial-qa-10K_test.parquet'))

In [ ]:
import pandas as pd

# If the dataset only has train, create a deterministic 80/20 split
if not test_parquet.exists():
    df = pd.read_parquet(train_parquet)

    # shuffle deterministically
    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

    cut = int(0.8 * len(df))
    df_train = df.iloc[:cut].copy()
    df_test  = df.iloc[cut:].copy()

    # overwrite train_parquet with the 80% train split (optional but recommended)
    df_train.to_parquet(train_parquet, index=False)
    df_test.to_parquet(test_parquet, index=False)

    print("Created splits:")
    print("train:", len(df_train), "->", train_parquet)
    print("test :", len(df_test),  "->", test_parquet)
else:
    print("Test parquet already exists:", test_parquet)


Test parquet already exists: /workspace/artifacts/benchmarks/virattt__financial-qa-10K_test.parquet


In [8]:
# Training or fine-tuning models using the selected method
print("train_parquet =", train_parquet, train_parquet.exists())
print("test_parquet  =", test_parquet,  test_parquet.exists())


train_parquet = /workspace/artifacts/benchmarks/virattt__financial-qa-10K_train.parquet True
test_parquet  = /workspace/artifacts/benchmarks/virattt__financial-qa-10K_test.parquet True


In [ ]:
# Builds retrieval corpus (chunks) from both splits (more coverage)
from workspace.data.build_corpus import build_corpus_from_parquet

CHUNKS_PATH = CORPUS_DIR / "chunks.jsonl"

build_corpus_from_parquet(
    parquet_paths=[str(train_parquet), str(test_parquet)],
    out_chunks_path=str(CHUNKS_PATH),
    chunk_size_chars=1800,
    chunk_overlap_chars=200,
    min_chars=200,
)

CHUNKS_PATH


2025-12-14 22:55:58,834 | data.build_corpus | INFO | Unique contexts collected: 2881
2025-12-14 22:55:58,867 | data.build_corpus | INFO | Wrote chunks: 2888 -> /workspace/artifacts/corpus/chunks.jsonl


PosixPath('/workspace/artifacts/corpus/chunks.jsonl')

In [ ]:
## Optional FAISS Code if you decide to use FAISS Indexing instead of BM25
## Note: If you use this code, you must uncomment this code and comment the code in the subsequent cell.

# from workspace.retrieval.faiss_index import build_faiss_index

# META_PATH = INDEX_DIR / "chunks_meta.jsonl"
# FAISS_PATH = INDEX_DIR / "faiss_index.bin"

# build_faiss_index(
#     chunks_path=str(CHUNKS_PATH),
#     meta_out_path=str(META_PATH),
#     index_out_path=str(FAISS_PATH),
#     embeddings_out_path=None,
#     model_name=EMBED_MODEL,
#     device="cuda",
#     batch_size=64,
#     normalize=True,
# )

# FAISS_PATH

In [ ]:
from workspace.retrieval.bm25_index import build_bm25_index
from workspace.retrieval.bm25_searcher import BM25Searcher

INDEX_DIR = ARTIFACTS / "indexes_bm25"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

META_PATH  = INDEX_DIR / "chunks_meta.jsonl"
BM25_PATH  = INDEX_DIR / "bm25_index.joblib"

build_bm25_index(
    chunks_path=str(CHUNKS_PATH),
    meta_out_path=str(META_PATH),
    index_out_path=str(BM25_PATH),
)



{'n_chunks': 2888,
 'meta_out': '/workspace/artifacts/indexes_bm25/chunks_meta.jsonl',
 'index_out': '/workspace/artifacts/indexes_bm25/bm25_index.joblib'}

In [12]:
# 4) Build evaluation benchmark JSONLs (train for DPO candidates, test for eval)
from workspace.data.build_benchmark import build_benchmark_from_parquet

TRAIN_JSONL = DATA_DIR / "train.jsonl"
TEST_JSONL = DATA_DIR / "test.jsonl"

build_benchmark_from_parquet(str(train_parquet), str(TRAIN_JSONL), max_examples=None)
build_benchmark_from_parquet(str(test_parquet), str(TEST_JSONL), max_examples=None)

TRAIN_JSONL, TEST_JSONL


2025-12-14 22:56:00,878 | data.build_benchmark | INFO | Saved benchmark 7000 rows -> /workspace/artifacts/benchmarks/train.jsonl
2025-12-14 22:56:00,974 | data.build_benchmark | INFO | Saved benchmark 1400 rows -> /workspace/artifacts/benchmarks/test.jsonl


(PosixPath('/workspace/artifacts/benchmarks/train.jsonl'),
 PosixPath('/workspace/artifacts/benchmarks/test.jsonl'))

In [ ]:
# Creates noisy (chat-style) benchmark queries

import json, random, re
from pathlib import Path

SEED = 42
random.seed(SEED)

def tokenize(s: str):
    return re.findall(r"[A-Za-z0-9%$]+|[^\sA-Za-z0-9]", s)

def detokenize(tokens):
    s = ""
    for t in tokens:
        if not s:
            s = t
        elif re.match(r"[A-Za-z0-9%$]", t) and re.match(r"[A-Za-z0-9%$]", s[-1]):
            s += " " + t
        elif t in [",", ".", "?", "!", ":", ";", "%"]:
            s += t
        else:
            s += " " + t
    return re.sub(r"\s+", " ", s).strip()

def add_typos(text: str, rate=0.05):
    words = text.split()
    out = []
    for w in words:
        if len(w) >= 5 and random.random() < rate and w.isalpha():
            i = random.randint(1, len(w)-2)
            w = w[:i] + w[i+1:]
        out.append(w)
    return " ".join(out)

DROP_PHRASES = [
    r"assume that you are.*?analyst\.?\s*",
    r"answer the following question.*?:\s*",
    r"answer in units of.*?\.\s*",
    r"round your answer to.*?\.\s*",
    r"define .*? as .*?\.\s*",
]

FINANCE_ANCHORS = [
    "working capital ratio", "net revenue", "operating income",
    "balance sheet", "income statement", "cash flow",
    "auditor", "critical audit matter", "segment", "cagr", "margin"
]

def make_noisy_question(clean_q: str, meta=None):
    q = clean_q

    # remove benchmark fluff
    for pat in DROP_PHRASES:
        q = re.sub(pat, "", q, flags=re.IGNORECASE)

    # drop years
    q = re.sub(r"\bFY?\s?\d{4}\b", "", q, flags=re.IGNORECASE)
    q = re.sub(r"\b20\d{2}\b", "", q)

    # drop finance anchors
    for a in FINANCE_ANCHORS:
        if random.random() < 0.4:
            q = re.sub(re.escape(a), "", q, flags=re.IGNORECASE)

    # shorten
    toks = tokenize(q)
    if len(toks) > 15:
        toks = toks[:random.randint(8, 14)]
    q = detokenize(toks)

    # casual + typos
    q = q.lower().replace("?", "")
    q = add_typos(q)

    q = re.sub(r"\s+", " ", q).strip()
    return q or clean_q.lower()

def build_noisy_benchmark(in_jsonl: str, out_jsonl: str):
    out_path = Path(out_jsonl)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(in_jsonl, "r", encoding="utf-8") as f_in, \
         open(out_jsonl, "w", encoding="utf-8") as f_out:
        for line in f_in:
            ex = json.loads(line)
            ex["question_noisy"] = make_noisy_question(ex["question"], ex.get("meta"))
            f_out.write(json.dumps(ex) + "\n")

    return out_path

TEST_NOISY_JSONL = DATA_DIR / "test_noisy.jsonl"
build_noisy_benchmark(str(TEST_JSONL), str(TEST_NOISY_JSONL))

print("Noisy benchmark written to:", TEST_NOISY_JSONL)


Noisy benchmark written to: /workspace/artifacts/benchmarks/test_noisy.jsonl


In [14]:
# Sanity check: benchmark rows include gold_doc_id (needed for Recall/MRR)
import json
with open(TEST_JSONL, "r", encoding="utf-8") as f:
    ex = json.loads(next(f))
ex.keys(), ex.get("gold_doc_id")

(dict_keys(['id', 'question', 'answer', 'gold_doc_id', 'meta']),
 'ctxsha1_35ed255aa4a839aca81c2224aa57ce557d496607')

In [ ]:
# Instantiates retrieval and corpus store
from workspace.retrieval.faiss_index import FaissSearcher
from workspace.retrieval.corpus_store import CorpusStore

# Use this if you prefer to use FAISS Indexing
# searcher = FaissSearcher(
#     index_path=str(FAISS_PATH),
#     meta_path=str(META_PATH),
#     embed_model=EMBED_MODEL,
#     device="cuda",
# )

# Use this if you prefer to use BM25 Indexing
searcher = BM25Searcher(index_path=str(BM25_PATH), meta_path=str(META_PATH))

store = CorpusStore(str(CHUNKS_PATH))


2025-12-14 22:56:04,129 | retrieval.corpus_store | INFO | Loaded chunk texts: 2888


In [16]:
# Loading data and resources required for this stage
!pip install -U "typing_extensions>=4.10" "pydantic>=2.7" "pydantic-core>=2.18" openai


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# Instantiates generator + rewriter clients (OpenAI or local)
from workspace.models.rag import RAGGenerator, QueryRewriter, AdaptiveRAG, RagConfig

if USE_OPENAI:
    from workspace.models.openai_client import OpenAIChat
    chat = OpenAIChat(model=OPENAI_MODEL)
else:
    from workspace.models.local_generator import LocalChat
    chat = LocalChat(model_name=LOCAL_GEN_MODEL)

generator = RAGGenerator(chat)
rewriter = QueryRewriter(chat)

cfg = RagConfig(
    top_k=20,
    context_k=5,
    confidence_threshold=0.8,
    max_adaptive_steps=3,
    alpha_gen=0.6,
)
rag = AdaptiveRAG(searcher=searcher, store=store, generator=generator, rewriter=rewriter, cfg=cfg)


In [ ]:
# Quick sanity check on a couple test questions
import json, itertools
from pprint import pprint

with open(TEST_NOISY_JSONL, "r", encoding="utf-8") as f:
    exs = [json.loads(next(f)) for _ in range(3)]

for ex in exs:
    res_b = rag.answer_baseline(ex["question_noisy"])
    res_a = rag.answer_adaptive(ex["question_noisy"])
    print("\nQ:", ex["question_noisy"])
    print("Gold:", ex["answer"])
    print("Baseline:", res_b["answer"], "conf=", res_b["confidence"], "triggered=", res_b["adaptive_triggered"])
    print("Adaptive :", res_a["answer"], "conf=", res_a["confidence"], "triggered=", res_a["adaptive_triggered"])
    if res_a["adaptive_triggered"]:
        print("Rewritten:", res_a["rewritten_query"])



Q: what criteria are used to classify loans and leases as nonperforming according
Gold: Loans and leases are classified as nonperforming when they are on nonaccrual status, such as being 90 days past due, have confirmed fraud or bankruptcy, or fit certain criteria such as being uninsured past a certain delinquency threshold or not well-secured and in the process of collection in the case of commercial loans.
Baseline: Loans and leases are classified as nonperforming if they are 90 days past due, placed on nonaccrual status, or have confirmed cases of fraud or bankruptcy. Specific types like consumer real estate-secured loans are nonperforming at 90 days past due unless fully insured, and commercial loans are nonperforming when past due 90 days or more unless well-secured and in the process of collection. conf= 0.9838723173313225 triggered= False
Adaptive : Loans and leases are classified as nonperforming if they are 90 days past due, placed on nonaccrual status, or have confirmed case

In [ ]:
# Quick sanity check on a couple test questions
import json, itertools
from pprint import pprint

with open(TEST_NOISY_JSONL, "r", encoding="utf-8") as f:
    exs = [json.loads(next(f)) for _ in range(3)]

for ex in exs:
    res_b = rag.answer_baseline(ex["question"])
    res_a = rag.answer_adaptive(ex["question"])
    print("\nQ:", ex["question"])
    print("Gold:", ex["answer"])
    print("Baseline:", res_b["answer"], "conf=", res_b["confidence"], "triggered=", res_b["adaptive_triggered"])
    print("Adaptive :", res_a["answer"], "conf=", res_a["confidence"], "triggered=", res_a["adaptive_triggered"])
    if res_a["adaptive_triggered"]:
        print("Rewritten:", res_a["rewritten_query"])



Q: What criteria are used to classify loans and leases as nonperforming according to the described credit policy?
Gold: Loans and leases are classified as nonperforming when they are on nonaccrual status, such as being 90 days past due, have confirmed fraud or bankruptcy, or fit certain criteria such as being uninsured past a certain delinquency threshold or not well-secured and in the process of collection in the case of commercial loans.
Baseline: Loans and leases are classified as nonperforming if they are 90 days past due, placed on nonaccrual status, or have confirmed cases of fraud or bankruptcy. Specific types like consumer real estate-secured loans are nonperforming at 90 days past due unless fully insured, and commercial loans are nonperforming when past due 90 days or more unless well-secured and in collection. conf= 0.9866741040030922 triggered= False
Adaptive : Loans and leases are classified as nonperforming if they are 90 days past due, placed on nonaccrual status, or ha

In [ ]:
# Overwrites all SEC-10K Questions to Noisy Versions
import json
from pathlib import Path

TEST_NOISY_FOR_EVAL = DATA_DIR / "test_noisy_as_question.jsonl"

with open(TEST_NOISY_JSONL, "r", encoding="utf-8") as fin, \
     open(TEST_NOISY_FOR_EVAL, "w", encoding="utf-8") as fout:
    for line in fin:
        ex = json.loads(line)
        ex["question"] = ex["question_noisy"]   # overwrite
        fout.write(json.dumps(ex) + "\n")

print("Wrote:", TEST_NOISY_FOR_EVAL)


Wrote: /workspace/artifacts/benchmarks/test_noisy_as_question.jsonl


In [ ]:
# Evaluates baseline and adaptive with base generator (full systems metrics)
from workspace.evaluation.eval_runner import run_eval

# summ_baseline = run_eval(
#     benchmark_jsonl=str(TEST_NOISY_JSONL),
#     answer_fn=rag.answer_baseline,
#     out_path=str(EVAL_DIR / "test_baseline_base_full.json"),
#     store=store, # enables SupportOverlap + evidence snippets
#     max_examples=100,
# )


# summ_adaptive = run_eval(
#     benchmark_jsonl=str(TEST_NOISY_JSONL),
#     answer_fn=rag.answer_adaptive,
#     out_path=str(EVAL_DIR / "test_adaptive_base_full.json"),
#     store=store,
#     max_examples=100,
# )

summ_baseline_noisy = run_eval(
    benchmark_jsonl=str(TEST_NOISY_FOR_EVAL),
    answer_fn=rag.answer_baseline,
    out_path=str(EVAL_DIR / "noisy_baseline.json"),
    store=store,
    max_examples=1, #Sets Number of Examples (Currently set to 1 for sanity purposes)
)

summ_adaptive_noisy = run_eval(
    benchmark_jsonl=str(TEST_NOISY_FOR_EVAL),
    answer_fn=rag.answer_adaptive,
    out_path=str(EVAL_DIR / "noisy_adaptive.json"),
    store=store,
    max_examples=1, #Sets Number of Examples (Currently set to 1 for sanity purposes)
)

summ_baseline_noisy, summ_adaptive_noisy


2025-12-14 22:56:30,606 | evaluation.eval_runner | INFO | Eval EM_norm=0.000 F1=0.513 ROUGE_L=0.389 SupportOverlap=0.000 Recall@5=1.000 Recall@10=1.000 Recall@20=1.000 MRR@20=1.000 Latency_p50_s=1.873 Throughput_qps=0.533 Adaptive_trigger_rate=0.000 Adaptive_avg_steps=0.000 (n=1)
2025-12-14 22:56:30,613 | evaluation.eval_runner | INFO | Saved eval -> /workspace/artifacts/eval_runs/noisy_baseline.json
2025-12-14 22:56:32,106 | evaluation.eval_runner | INFO | Eval EM_norm=0.000 F1=0.513 ROUGE_L=0.389 SupportOverlap=0.000 Recall@5=1.000 Recall@10=1.000 Recall@20=1.000 MRR@20=1.000 Latency_p50_s=1.486 Throughput_qps=0.671 Adaptive_trigger_rate=0.000 Adaptive_avg_steps=0.000 (n=1)
2025-12-14 22:56:32,122 | evaluation.eval_runner | INFO | Saved eval -> /workspace/artifacts/eval_runs/noisy_adaptive.json


({'n': 1,
  'EM_norm': 0.0,
  'F1': 0.5132743362831859,
  'ROUGE_L': 0.3893805309734513,
  'SupportOverlap': 0.0,
  'Recall@5': 1.0,
  'Recall@10': 1.0,
  'Recall@20': 1.0,
  'MRR@20': 1.0,
  'Latency_p50_s': 1.8733515925705433,
  'Latency_p95_s': None,
  'Throughput_qps': 0.5329672064872968,
  'Adaptive_trigger_rate': 0.0,
  'Adaptive_avg_steps': 0.0},
 {'n': 1,
  'EM_norm': 0.0,
  'F1': 0.5132743362831859,
  'ROUGE_L': 0.3893805309734513,
  'SupportOverlap': 0.0,
  'Recall@5': 1.0,
  'Recall@10': 1.0,
  'Recall@20': 1.0,
  'MRR@20': 1.0,
  'Latency_p50_s': 1.48618645966053,
  'Latency_p95_s': None,
  'Throughput_qps': 0.6713162178761506,
  'Adaptive_trigger_rate': 0.0,
  'Adaptive_avg_steps': 0.0})

In [23]:
# Applying adaptive query rewriting based on confidence signals
GOOD_Q = "What environmental commitment did Hasbro make for reducing greenhouse gas emissions by 2030?"
BAD_Q  = "hasbro's financial"

def pretty(res, label):
    print("\n" + "="*80)
    print(label)
    print("Q:", res["question"])
    if res.get("rewritten_query"):
        print("Rewritten:", res["rewritten_query"])
    print("Confidence:", round(res.get("confidence", -1), 3),
          "| Gen:", round(res.get("gen_conf", -1), 3),
          "| Retr:", round(res.get("retr_conf", -1), 3))
    top1 = res["hits"][0]["score"] if res.get("hits") else None
    print("Top1 retrieval score:", top1)
    print("Answer:", res["answer"][:600])
    print("\nTop evidence snippet:")
    if res.get("hits"):
        h = res["hits"][0]
        txt = store.get_text(h["doc_id"], h["chunk_id"])
        print(txt[:500])

# Baseline vs adaptive on BAD query
bad_base = rag.answer_baseline(BAD_Q)
bad_adap = rag.answer_adaptive(BAD_Q)

pretty(bad_base, "BAD QUERY — baseline")
pretty(bad_adap, "BAD QUERY — adaptive")

# Baseline on GOOD query (for reference)
good_base = rag.answer_baseline(GOOD_Q)
pretty(good_base, "GOOD QUERY — baseline (reference)")


2025-12-14 22:56:33,842 | models.rag | INFO | Adaptive step 1: 'hasbro's financial' -> 'Hasbro's financial performance and metrics.'
2025-12-14 22:56:35,099 | models.rag | INFO | Adaptive step 2: 'Hasbro's financial performance and metrics.' -> 'hasbro's financial'
2025-12-14 22:56:36,096 | models.rag | INFO | Adaptive step 3: 'hasbro's financial' -> 'Hasbro's financial performance and metrics.'

BAD QUERY — baseline
Q: hasbro's financial
Confidence: 0.224 | Gen: 0.0 | Retr: 0.56
Top1 retrieval score: 9.61830876464337
Answer: CANNOT_ANSWER

Top evidence snippet:
On April 12, 2023, we announced the appointment of Gina Goetter as Chief Financial Officer, effective May 18, 2023. Ms. Goetter joined Hasbro from Harley Davidson, Inc., where she served as Chief Financial Officer.

BAD QUERY — adaptive
Q: hasbro's financial
Rewritten: Hasbro's financial performance and metrics.
Confidence: 0.309 | Gen: 0.0 | Retr: 0.772
Top1 retrieval score: 16.130986986429523
Answer: CANNOT_ANSWER

Top eviden

In [ ]:
import json
path = EVAL_DIR / "test_adaptive_base_full.json"
data = json.load(open(path))
items = data["items"] if isinstance(data, dict) and "items" in data else data
print("n_items:", len(items))


n_items: 2


## Configuration and Argument Definitions

## Optional: DPO alignment (local generator only)

If you use OpenAI for generation, you can still use OpenAI as a **judge**, but DPO training produces a **local** LoRA adapter to apply to the local HF model.

If `USE_OPENAI=True`, you can still run DPO *provided you have a local model to train*. The code below uses `LOCAL_GEN_MODEL`.


In [27]:
# Training or fine-tuning models using the selected method
DO_DPO = True  # set False to skip


In [ ]:
#Generates candidate answer pairs from TRAIN set
if DO_DPO:
    from workspace.rlhf.gen_candidates import gen_candidates

    CANDS_PATH = PREFS_DIR / "candidates.jsonl"
    gen_candidates(
        dataset_jsonl=str(TRAIN_JSONL),
        searcher=searcher,
        store=store,
        chat_client=chat,
        out_path=str(CANDS_PATH),
        n=100,
        top_k=20,
        context_k=5,
    )
    CANDS_PATH


2025-12-14 22:59:30,682 | rlhf.gen_candidates | INFO | Wrote 100 candidate pairs -> /workspace/artifacts/prefs/candidates.jsonl


In [ ]:
# Judges pairs to create DPO dataset
if DO_DPO:
    from workspace.rlhf.judge_pairs import judge_pairs

    if USE_OPENAI:
        # use OpenAI as judge
        judge_client = chat
    else:
        # local judge (weaker); still works but noisier
        judge_client = chat

    DPO_DATA = PREFS_DIR / "dpo_dataset.jsonl"
    judge_pairs(
        candidates_jsonl=str(CANDS_PATH),
        judge_client=judge_client,
        out_dpo_jsonl=str(DPO_DATA),
        max_pairs=5,
    )
    DPO_DATA


2025-12-14 22:59:40,034 | rlhf.judge_pairs | INFO | Judged 23 pairs, kept 5 -> /workspace/artifacts/prefs/dpo_dataset.jsonl


In [ ]:
import trl, transformers, peft
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("peft", peft.__version__)


trl 0.26.1
transformers 4.57.3
peft 0.18.0


## Data Loading and Preprocessing




In [ ]:
# Training DPO LoRA adapter
if DO_DPO:
    from workspace.rlhf.train_dpo_lora import train_dpo_lora

    DPO_OUT = MODELS_DIR / "dpo_lora_v1"
    train_dpo_lora(
        dpo_jsonl=str(DPO_DATA),
        base_model_name=LOCAL_GEN_MODEL,
        output_dir=str(DPO_OUT),
        max_rows=200,
        batch_size=2,
        grad_accum=8,
        lr=1e-5,
        epochs=1,
        beta=0.1,
    )
    DPO_OUT


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Extracting prompt in train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:92: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss


In [ ]:
# Evaluating baseline/adaptive with DPO-aligned generator (local)
    
if DO_DPO:
    from workspace.models.local_generator import LocalChat
    from workspace.models.rag import RAGGenerator, QueryRewriter, AdaptiveRAG, RagConfig

    chat_dpo = LocalChat(model_name=LOCAL_GEN_MODEL, lora_path=str(DPO_OUT))
    generator_dpo = RAGGenerator(chat_dpo)
    # rewriting can still use OpenAI or base local; keep same as before
    rewriter_same = QueryRewriter(chat)

    rag_dpo = AdaptiveRAG(
        searcher=searcher,
        store=store,
        generator=generator_dpo,
        rewriter=rewriter_same,
        cfg=cfg,
    )

    summ_baseline_noisy = run_eval(
        benchmark_jsonl=str(TEST_NOISY_FOR_EVAL),
        answer_fn=rag_dpo.answer_baseline,
        out_path=str(EVAL_DIR / "noisy_baseline_dpo.json"),
        max_examples=100,
    )
    
    summ_adaptive_noisy = run_eval(
        benchmark_jsonl=str(TEST_NOISY_FOR_EVAL),
        answer_fn=rag_dpo.answer_adaptive,
        out_path=str(EVAL_DIR / "noisy_adaptive_dpo.json"),
        max_examples=100,
    )

    summ_baseline_dpo, summ_adaptive_dpo


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2025-12-14 22:59:50,233 | models.local_generator | INFO | Loading LoRA adapters from /workspace/artifacts/models/dpo_lora_v1


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2025-12-14 23:10:14,518 | evaluation.eval_runner | INFO | Eval EM_norm=0.030 F1=0.206 ROUGE_L=0.192 Recall@5=0.350 Recall@10=0.400 Recall@20=0.410 MRR@20=0.301 Latency_p50_s=6.198 Latency_p95_s=6.557 Throughput_qps=0.160 Adaptive_trigger_rate=0.000 Adaptive_avg_steps=0.000 (n=100)
2025-12-14 23:10:14,529 | evaluation.eval_runner | INFO | Saved eval -> /workspace/artifacts/eval_runs/noisy_baseline_dpo.json
2025-12-14 23:10:21,465 | models.rag | INFO | Adaptive step 1: 'what criteria are used to classify loans and leases as nonperforming according' -> 'what criteria are used to classify loans and leases as nonperforming according'
2025-12-14 23:10:28,805 | models.rag | INFO | Adaptive step 2: 'what criteria are used to classify loans and leases as nonperforming according' -> 'What criteria classify loans and leases as nonperforming, specifically regarding nonaccrual status and delinquency periods?'
2025-12-14 23:10:35,699 | models.rag | INFO | Adaptive step 3: 'What criteria classify loa

NameError: name 'summ_baseline_dpo' is not defined

In [33]:
# Applying adaptive query rewriting based on confidence signals
summ_baseline_noisy, summ_adaptive_noisy

({'n': 100,
  'EM_norm': 0.03,
  'F1': 0.2058229785471893,
  'ROUGE_L': 0.192450166936231,
  'SupportOverlap': None,
  'Recall@5': 0.35,
  'Recall@10': 0.4,
  'Recall@20': 0.41,
  'MRR@20': 0.3005952380952381,
  'Latency_p50_s': 6.197540912777185,
  'Latency_p95_s': 6.556627913750708,
  'Throughput_qps': 0.16026372876026165,
  'Adaptive_trigger_rate': 0.0,
  'Adaptive_avg_steps': 0.0},
 {'n': 100,
  'EM_norm': 0.04,
  'F1': 0.21334850435037733,
  'ROUGE_L': 0.19861788460826188,
  'SupportOverlap': None,
  'Recall@5': 0.39,
  'Recall@10': 0.42,
  'Recall@20': 0.42,
  'MRR@20': 0.31069047619047613,
  'Latency_p50_s': 27.188821248710155,
  'Latency_p95_s': 28.54773761127144,
  'Throughput_qps': 0.040688883414180047,
  'Adaptive_trigger_rate': 0.9,
  'Adaptive_avg_steps': 2.8777777777777778})